# ADP1 DGOA Analysis

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

import cobra
from modelseedpy import MSModelUtil

# Load the published model and translate gene IDs using KBase genome
pubmod = MSModelUtil.from_cobrapy("models/PublishedModel.XML")
_legacy.load_kbase_gene_container("Acinetobacter_baylyi_ADP1_RPG_Snekmer_LA_Pfam", ws=219173, localname="ADP1")
translation = {}
for gene in pubmod.model.genes:
    ftrs = _legacy.alias_to_ftrs("ADP1", gene.id)
    if len(ftrs) > 0:
        translation[gene.id] = ftrs[0]
session.cache.save("gene_translation", translation)

from cobra.manipulation.modify import rename_genes
rename_genes(pubmod.model, translation)
cobra.io.save_json_model(pubmod.model, "models/TranslatedPublishedModel.json")
print(f"Translated {len(translation)} gene IDs and saved model")


## Merge annotation data

In [ ]:
%run util.py

gene_data = session.cache.load("gene_term_hash_named")
gene_model_data = session.cache.load("genedata")
for geneid in gene_model_data:
    if geneid in gene_data:
        for item in gene_data[geneid]:
            if item[0:4] == "RAST":
                gene_model_data[geneid]["RAST"] = gene_data[geneid][item]
            if item[0:6] == "Prokka":
                gene_model_data[geneid]["Prokka"] = gene_data[geneid][item]
            if item[0:4] == "DRAM":
                gene_model_data[geneid]["DRAM"] = gene_data[geneid][item]
session.cache.save("genedata", gene_model_data)
print(f"Merged annotation data for {len(gene_model_data)} genes")


## ModelSEED model simulation

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

from cobra.flux_analysis import pfba

# Load model and set media via _legacy
model = _legacy.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
pyruvate_media = _legacy.get_media("KBaseMedia/Carbon-Pyruvic-Acid")
solution = _legacy.run_fba(model, media=pyruvate_media, objective="bio1", run_pfba=True)
ms_output = _legacy.run_fva(model, fraction_of_optimum=0.9)
session.cache.save("ms_fva", ms_output)

rxn_fluxes = {}
for rxn in model.model.reactions:
    if abs(solution.fluxes[rxn.id]) > 1e-9:
        rxn_fluxes[rxn.id] = solution.fluxes[rxn.id]

session.cache.save("ms_active_fluxes", rxn_fluxes)
print(f"ModelSEED model: {len(rxn_fluxes)} active reactions, biomass={solution.fluxes.get('bio1', 0):.4f}")


## Model namespace translation

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

from modelseedpy import MSModelUtil

rxn_data = session.cache.load("reactiondata")
pubmod = MSModelUtil.from_cobrapy("models/PublishedModel.XML")
msmodel = _legacy.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
[model, testoutput] = _legacy.translate_model_to_ms_namespace(pubmod, msmodel)
session.cache.save("testoutput", testoutput)
_legacy.add_model_translation_to_data(model, rxn_data)
session.cache.save("reactiondata", rxn_data)
print("Model namespace translation complete")


## Published model simulation with DGOA

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

from cobra.io import load_json_model
from cobra.flux_analysis import pfba
from modelseedpy import MSModelUtil

pubmod = MSModelUtil.from_cobrapy("models/TranslatedPublishedModel.json")
pyruvate_media = _legacy.get_media("KBaseMedia/Carbon-Pyruvic-Acid")
solution = _legacy.run_fba(pubmod, media=pyruvate_media, objective="bio1", run_pfba=True)

# Knock out existing DAHP reaction
pubmod.model.reactions.get_by_id("rxn01332_c0").lower_bound = 0
pubmod.model.reactions.get_by_id("rxn01332_c0").upper_bound = 0
pubmod_output = _legacy.run_fva(pubmod, fraction_of_optimum=0.9)
session.cache.save("pubmod_fva", pubmod_output)

# Add DGOA reaction and re-simulate
_legacy.add_dgoa_reaction(pubmod)
solution2 = _legacy.run_fba(pubmod, media=pyruvate_media, objective="bio1", run_pfba=True)
pubmod_dgoa_output = _legacy.run_fva(pubmod, fraction_of_optimum=0.9)
session.cache.save("pubmod_dgoa_fva", pubmod_dgoa_output)

print(f"Published model biomass (no DGOA): {solution.fluxes.get('bio1', solution.objective_value):.4f}")
print(f"Published model biomass (with DGOA): {solution2.fluxes.get('bio1', solution2.objective_value):.4f}")


## Single-reaction knockout analysis

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

from cobra.flux_analysis import pfba
import cobra
from modelseedpy import MSModelUtil

# Load the model with DGOA
model = _legacy.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
_legacy.add_dgoa_reaction(model)

# Set up media
pyruvate_media = _legacy.get_media("KBaseMedia/Carbon-Pyruvic-Acid")
_legacy.set_media(model, pyruvate_media)
model.model.objective = "bio1"

records = []
wt_solution = pfba(model.model)
for rxn in model.model.reactions:
    if abs(wt_solution.fluxes[rxn.id]) > 1e-9:
        old_lb = rxn.lower_bound
        old_ub = rxn.upper_bound
        rxn.lower_bound = 0
        rxn.upper_bound = 0
        ko_sol = pfba(model.model)
        rxn.lower_bound = old_lb
        rxn.upper_bound = old_ub
        records.append({
            "reaction": rxn.id,
            "wt_flux": wt_solution.fluxes[rxn.id],
            "ko_biomass": ko_sol.fluxes.get("bio1", 0),
            "wt_biomass": wt_solution.fluxes.get("bio1", 0),
            "ko_ratio": ko_sol.fluxes.get("bio1", 0) / wt_solution.fluxes.get("bio1", 1),
        })

ko_df = pd.DataFrame(records)
session.cache.save("dgoa_ko_analysis", ko_df.to_dict(orient="records"))
print(f"Knockout analysis: {len(records)} active reactions analyzed")
print(f"WT biomass: {wt_solution.fluxes.get('bio1', 0):.4f}")


## Multi-condition DGOA analysis

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

from cobra.flux_analysis import pfba
import cobra
from modelseedpy import MSModelUtil

# Load the model
model = _legacy.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
_legacy.add_dgoa_reaction(model)

# Set up media
pyruvate_media = _legacy.get_media("KBaseMedia/Carbon-Pyruvic-Acid")
_legacy.set_media(model, pyruvate_media)
model.model.objective = "bio1"

# 1. Wild type analysis - rxn01332 active, DGOA inactive
model.model.reactions.get_by_id("DgoA").lower_bound = 0
model.model.reactions.get_by_id("DgoA").upper_bound = 0
wt_sol = pfba(model.model)
wt_fluxes = {rxn.id: wt_sol.fluxes[rxn.id] for rxn in model.model.reactions if abs(wt_sol.fluxes[rxn.id]) > 1e-9}

# 2. DGOA active, rxn01332 knocked out
model.model.reactions.get_by_id("DgoA").lower_bound = 0
model.model.reactions.get_by_id("DgoA").upper_bound = 1000
model.model.reactions.get_by_id("rxn01332_c0").lower_bound = 0
model.model.reactions.get_by_id("rxn01332_c0").upper_bound = 0
dgoa_sol = pfba(model.model)
dgoa_fluxes = {rxn.id: dgoa_sol.fluxes[rxn.id] for rxn in model.model.reactions if abs(dgoa_sol.fluxes[rxn.id]) > 1e-9}

# Compare
results = {
    "wt_biomass": wt_sol.fluxes.get("bio1", 0),
    "dgoa_biomass": dgoa_sol.fluxes.get("bio1", 0),
    "wt_active_rxns": len(wt_fluxes),
    "dgoa_active_rxns": len(dgoa_fluxes),
    "wt_fluxes": wt_fluxes,
    "dgoa_fluxes": dgoa_fluxes,
}
session.cache.save("dgoa_condition_comparison", results)

print(f"WT: biomass={results['wt_biomass']:.4f}, active_rxns={results['wt_active_rxns']}")
print(f"DGOA: biomass={results['dgoa_biomass']:.4f}, active_rxns={results['dgoa_active_rxns']}")


## Load cluster sizes

In [ ]:
%run util.py

# Read the ClusterSizes.txt file into a dictionary
cluster_sizes = {}
with open("data/ClusterSizes.txt", "r") as file:
    for line in file:
        cluster_id, size = line.strip().split("\t")
        cluster_sizes[cluster_id] = int(size)

session.cache.save("cluster_sizes", cluster_sizes)
print(f"Loaded {len(cluster_sizes)} cluster sizes")


## Escher map visualization

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

from modelseedpy import MSModelUtil

# Load model and flux data
model = _legacy.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
pyruvate_media = _legacy.get_media("KBaseMedia/Carbon-Pyruvic-Acid")
solution = _legacy.run_fba(model, media=pyruvate_media, objective="bio1", run_pfba=True)

# Create flux dictionary for Escher
rxn_fluxes = {}
for rxn in model.model.reactions:
    if abs(solution.fluxes[rxn.id]) > 1e-9:
        rxn_fluxes[rxn.id] = solution.fluxes[rxn.id]

# Generate Escher map using _legacy
try:
    output = _legacy.create_map_html2(
        model=model.model,
        flux=pd.Series(rxn_fluxes),
        map="Core",
        output_path="nboutput/escher_dgoa_core.html",
    )
    print(f"Escher map created: nboutput/escher_dgoa_core.html")
except Exception as e:
    print(f"Escher map generation failed (non-critical): {e}")
